# 410 · SFT 데이터셋 구축

**CPU로 실행 가능**

## 이 노트북의 산출물
```
artifacts/data/train.jsonl        <- 420 학습 입력
artifacts/data/eval.jsonl         <- 430 평가 입력 (학습에 절대 쓰이지 않음)
artifacts/data/domain30.jsonl     <- 수강생이 직접 작성 (430 정성 평가용)
artifacts/data/data_report.json   <- 품질 5항목 점검 결과
```

## 학습목표
1. 목적에 맞는 데이터 규모·품질 기준을 정할 수 있다
2. `apply_chat_template`으로 대상 모델 형식에 맞게 포맷할 수 있다
3. **손실 마스킹(-100)을 적용하고 결과를 눈으로 검증**할 수 있다
4. 중복·누수·길이 이상치·형식 불일치·평가셋 오염을 점검할 수 있다


In [1]:
# --- 패키지 설치 (Colab) ---
# 설치 후 "런타임 다시 시작" 안내가 나오면 재시작하고, 다음 셀부터 이어서 실행한다.
!pip install -q transformers==4.57.1 datasets==4.0.0 accelerate==1.10.1 \
    peft==0.17.1 trl==0.23.0 bitsandbytes==0.47.0 tiktoken==0.11.0
!apt-get install -y fonts-nanum -qq > /dev/null   # matplotlib 한글 폰트

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# --- 저장소 클론 및 루트 이동 (Colab) ---
from pathlib import Path

if not Path("common/config.py").exists():
    !git clone -q https://github.com/ironmanciti/LLM_FineTuning_Lecture.git
    %cd LLM_FineTuning_Lecture
print("저장소 루트:", Path.cwd())

from common import config as C, env, artifacts as art
env.set_seed(C.SEED)

/content/LLM_FineTuning_Lecture
저장소 루트: /content/LLM_FineTuning_Lecture


42

In [ ]:
# --- Google Drive 연결: artifacts/ 영속화 (Colab) ---
# 노트북마다 런타임(VM)이 다르므로, 산출물을 Drive에 두어야 다음 노트북에서 읽을 수 있다.
import shutil
from pathlib import Path

try:
    from google.colab import drive  # Colab이 아니면 ImportError

    drive.mount("/content/drive")

    DRIVE_ART = Path("/content/drive/MyDrive/LLM_FineTuning_Lecture/artifacts")
    DRIVE_ART.mkdir(parents=True, exist_ok=True)

    local_art = Path("artifacts")
    if not local_art.is_symlink():
        if local_art.exists():  # 이미 로컬에 쓴 결과가 있으면 Drive로 이관
            shutil.copytree(local_art, DRIVE_ART, dirs_exist_ok=True)
            shutil.rmtree(local_art)
        local_art.symlink_to(DRIVE_ART)
    print("artifacts ->", local_art.resolve())
except ImportError:
    print("Colab이 아니므로 로컬 artifacts/ 를 그대로 사용합니다.")


---
## 1. 태스크 정의 — 무엇을 가르칠 것인가

이 실습의 태스크는 **지문 기반 질의응답을 JSON으로 출력하기**다.

왜 JSON인가? 0.5B급 소형 모델에서 **EM/F1은 잘 오르지 않지만 형식 준수율은 뚜렷하게
오른다**(2일차 §3.5-4). 파인튜닝의 효과를 정직하게 관측할 수 있는 지표를 태스크 설계에
미리 심어 둔 것이다. 3일차 `430`에서 이 설계가 회수된다.

| 항목 | 값 |
|---|---|
| 입력 | 지문 + 질문 |
| 출력 | `{"answer": "...", "evidence": "..."}` |
| 목적 | **형식 교정 + 도메인 톤** (지식 주입이 아니다 — §3.7-3) |
| 규모 | 수천 건 (형식 교정은 수백 건이면 되지만, 지표 변화를 보려면 더 필요) |

In [3]:
print(C.summary())
print("\nsystem 프롬프트:")
print(" ", C.SYSTEM_PROMPT)
print("\n필수 필드:", C.REQUIRED_FIELDS)

모델        : Qwen/Qwen2.5-0.5B-Instruct
데이터      : KorQuAD/squad_kor_v1  (train 2000 / eval 200 / domain 30)
max_seq_len : 768
LoRA        : r=16 alpha=32 target=all-linear
유효 배치   : 1 x 8 = 8
LR / epochs : 0.0002 / 2
seed        : 42

system 프롬프트:
  당신은 주어진 지문에서만 근거를 찾아 답하는 한국어 질의응답 도우미입니다. answer와 evidence 두 개의 키를 가진 JSON 객체만 출력하십시오. 설명이나 인사말을 덧붙이지 마십시오.

필수 필드: ['answer', 'evidence']


In [4]:
# 020에서 정한 max_seq_len을 읽어온다. 없으면 기본값.
tc = art.load_json(art.token_cost_path(), default=None)
if tc:
    MAX_LEN = tc["recommended_max_seq_len"]
    print(f"020의 권고값을 사용합니다: max_seq_len = {MAX_LEN}")
    print(f"  (p95 = {tc['length_stats']['p95']:.0f}, 측정 표본 {tc['n_samples_measured']}건)")
else:
    MAX_LEN = C.MAX_SEQ_LEN
    print(f"[주의] artifacts/tokens/token_cost.json 이 없습니다. 020을 먼저 실행하십시오.")
    print(f"       기본값 {MAX_LEN} 으로 진행합니다.")

[주의] artifacts/tokens/token_cost.json 이 없습니다. 020을 먼저 실행하십시오.
       기본값 768 으로 진행합니다.


---
## 2. 원천 데이터 로드와 정제

구 `050-Data_Preprocessing` 노트북의 정제 코드를 이 단계로 흡수했다.
정제는 독립 주제가 아니라 **데이터셋 구축의 한 단계**다.

In [5]:
import json, re, unicodedata, hashlib, random
from datasets import load_dataset

random.seed(C.SEED)

raw = load_dataset(C.DATASET_ID)
print(raw)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 60407
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5774
    })
})


In [6]:
def clean_text(s: str) -> str:
    """정제 — 무엇을 지우고 무엇을 남기는가.

    LLM 시대의 정제는 전통 NLP와 방향이 반대다.
    불용어 제거·어간 추출은 **하지 않는다**(모델이 문맥을 쓰므로 정보를 버리는 손해).
    지우는 것은 노이즈뿐이다.
    """
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("\u200b", "").replace("\xa0", " ")
    # 태그를 공백으로 바꾸면 한국어 교착이 깨진다("<b>서울</b>은" -> "서울 은").
    # 빈 문자열로 지우는 것이 한국어에서는 더 안전하다.
    s = re.sub(r"<[^>]+>", "", s)            # 잔여 HTML 태그
    s = re.sub(r"\[\d+\]", "", s)           # 위키 각주 [1]
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

# 확인: 무엇이 바뀌는가
demo = "  <b>서울</b>은\u200b 대한민국의   수도이다.[12]\n\n\n\n다음 문장. "
print(repr(demo)); print(repr(clean_text(demo)))

'  <b>서울</b>은\u200b 대한민국의   수도이다.[12]\n\n\n\n다음 문장. '
'서울은 대한민국의 수도이다.\n\n다음 문장.'


In [7]:
def to_records(split, n):
    out = []
    for r in split.select(range(min(n, len(split)))):
        ctx = clean_text(r["context"])
        q = clean_text(r["question"])
        answers = [clean_text(a) for a in r["answers"]["text"] if a and a.strip()]
        if not (ctx and q and answers):
            continue
        ans = answers[0]
        # evidence = 정답을 포함하는 문장. 근거를 함께 내도록 가르치면
        # 모델이 지문 밖에서 답을 만들어내는 경향이 줄어든다.
        sents = re.split(r"(?<=[.!?])\s+", ctx)
        ev = next((s for s in sents if ans in s), ctx[:200])
        out.append({
            "id": r.get("id", hashlib.md5((ctx + q).encode()).hexdigest()[:12]),
            "context": ctx,
            "question": q,
            "answer": ans,
            "answer_all": "||".join(dict.fromkeys(answers)),
            "evidence": ev.strip(),
        })
    return out

train_rec = to_records(raw["train"], C.N_TRAIN * 2)     # 여유 있게 뽑고 점검에서 걸러낸다
eval_rec = to_records(raw["validation"], C.N_EVAL * 2)
print(f"정제 후: train 후보 {len(train_rec)}건 / eval 후보 {len(eval_rec)}건")
print("\n예시:")
print(json.dumps(train_rec[0], ensure_ascii=False, indent=2)[:600])

정제 후: train 후보 4000건 / eval 후보 400건

예시:
{
  "id": "6566495-0-0",
  "context": "1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 경우에도 그의 전기에 적혀 있는 것처럼 단순한 정신적 피로나 실의가 반영된 것이 아니라 베토벤의 합창교향곡 조성의 영향을 받은 것을 볼 수 있다. 그렇게 교향곡 작곡을 1839년부터 40년에 걸쳐 파리에서 착수했으나 1악장을 쓴 뒤에 중단했다. 또한 작품의 완성과 동시에 그는 이 서곡(1악장)을 파리 음악원의 연주회에서 연주할 파트보까지 준비하였으나, 실제로는 이루어지지는 않았다. 결국 초연은 4년 반이 지난 후에 드레스덴에서 연주되었고 재연도 이루어졌지만, 이후


---
## 3. chat template 적용 — 문자열을 손으로 조립하지 않는다

같은 데이터라도 **모델마다 특수 토큰이 다르다.** `<|im_start|>`를 직접 쓰면 모델이
바뀔 때 조용히 깨진다. 반드시 토크나이저에게 물어야 한다.

In [8]:
from transformers import AutoTokenizer
from common import chat

tok = AutoTokenizer.from_pretrained(C.MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def make_target(rec) -> str:
    """모델이 출력해야 하는 문자열. 공백까지 일관되게 고정한다."""
    return json.dumps({"answer": rec["answer"], "evidence": rec["evidence"]},
                      ensure_ascii=False)

rec = train_rec[0]
msgs = chat.build_messages(rec["question"], system=C.SYSTEM_PROMPT, context=rec["context"])
prompt = chat.render_prompt(tok, msgs)

print("=" * 70)
print("템플릿 적용 전 (우리가 가진 것)")
print("=" * 70)
print(f"system   : {C.SYSTEM_PROMPT[:60]}...")
print(f"user     : [지문 {len(rec['context'])}자] + {rec['question']}")
print(f"assistant: {make_target(rec)}")
print()
print("=" * 70)
print("템플릿 적용 후 (모델이 보는 것)")
print("=" * 70)
print(prompt[:400] + ("..." if len(prompt) > 400 else ""))
print()
print("raw:", repr(prompt[:180]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

템플릿 적용 전 (우리가 가진 것)
system   : 당신은 주어진 지문에서만 근거를 찾아 답하는 한국어 질의응답 도우미입니다. answer와 evidence 두...
user     : [지문 673자] + 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?
assistant: {"answer": "교향곡", "evidence": "1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다."}

템플릿 적용 후 (모델이 보는 것)
<|im_start|>system
당신은 주어진 지문에서만 근거를 찾아 답하는 한국어 질의응답 도우미입니다. answer와 evidence 두 개의 키를 가진 JSON 객체만 출력하십시오. 설명이나 인사말을 덧붙이지 마십시오.<|im_end|>
<|im_start|>user
1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 ...

raw: '<|im_start|>system\n당신은 주어진 지문에서만 근거를 찾아 답하는 한국어 질의응답 도우미입니다. answer와 evidence 두 개의 키를 가진 JSON 객체만 출력하십시오. 설명이나 인사말을 덧붙이지 마십시오.<|im_end|>\n<|im_start|>user\n1839년 바그너는 괴테의 파우스트을 처음 읽고'


### 학습용과 추론용의 차이

- **학습**: 프롬프트 + 정답 + EOS 를 하나의 시퀀스로 만든다
- **추론**: 프롬프트 + `add_generation_prompt=True`(생성 시작 토큰)까지만

`common/chat.py`가 두 경로에 **같은 템플릿**을 쓰도록 묶어 두었다. 이것이 어긋나면
학습 효과가 사라진다 — 가장 흔한 실패 원인이다.

---
## 4. 손실 마스킹 — 눈으로 확인한다

프롬프트 토큰까지 손실에 넣으면 모델이 **질문을 따라 쓰는 법**을 배운다.
프롬프트 구간의 라벨을 `-100`으로 덮어 손실에서 제외한다.

아래 셀은 그 결과를 **색으로** 출력한다. 숫자 배열로는 아무도 확인하지 않는다.

In [9]:
ex = chat.build_example(tok, msgs, make_target(rec), max_len=MAX_LEN, mask_prompt=True)

print(f"총 {ex['n_total']}토큰 = 프롬프트 {ex['n_prompt']} + 정답 {ex['n_answer']}")
print(f"잘림 여부: {ex['truncated']}\n")
print(chat.render_mask(tok, ex, max_tokens=320))

총 661토큰 = 프롬프트 590 + 정답 71
잘림 여부: False

■ 회색 = 손실 제외(프롬프트, 590토큰)   ■ 파랑 = 학습됨(정답+EOS, 71토큰)

... (이하 341 토큰 생략)<|im_start|>system
당신은 주어진 지문에서만 근거를 찾아 답하는 한국어 질의응답 도우미입니다. answer와 evidence 두 개의 키를 가진 JSON 객체만 출력하십시오. 설명이나 인사말을 덧붙이지 마십시오.<|im_end|>
<|im_start|>user
1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 경우에도 그의 전기에 적혀 있는 것처럼 단순한 정신적


### 확인할 것 세 가지

1. **특수 토큰이 회색 구간에 정상적으로 들어갔는가**
2. **정답(JSON)만 파랑인가**
3. **파랑 구간 끝에 EOS가 있는가** — 없으면 모델이 멈추는 법을 못 배운다
   → 3일차 `430`의 `clean_stop_pct`가 낮게 나온다

In [10]:
chk = chat.check_example(tok, ex)
print("자동 점검:", "통과" if chk["ok"] else "문제 있음")
print(f"  학습되는 토큰 {chk['trainable_tokens']} / 전체 {chk['total_tokens']} "
      f"({100*chk['trainable_tokens']/chk['total_tokens']:.1f}%)")
for p in chk["problems"]:
    print("  [!]", p)
assert chk["ok"], "마스킹에 문제가 있습니다. 위 메시지를 확인하십시오."

자동 점검: 통과
  학습되는 토큰 71 / 전체 661 (10.7%)


In [11]:
# 마스킹 유/무 비교 — 왜 마스킹하는가 (§3.9-4)
ex_nomask = chat.build_example(tok, msgs, make_target(rec), max_len=MAX_LEN, mask_prompt=False)
c2 = chat.check_example(tok, ex_nomask)

print(f"{'설정':<14}{'학습 토큰':>10}{'비율':>8}  진단")
print(f"{'마스킹 O':<14}{chk['trainable_tokens']:>10}{100*chk['trainable_tokens']/chk['total_tokens']:>7.1f}%  정답만 학습")
print(f"{'마스킹 X':<14}{c2['trainable_tokens']:>10}{100*c2['trainable_tokens']/c2['total_tokens']:>7.1f}%  {c2['problems'][0] if c2['problems'] else ''}")
print("\n마스킹을 안 하면 손실의 대부분이 '지문을 그대로 다시 쓰는 법'에서 나온다.")
print("손실 곡선은 더 예쁘게 내려가지만 원하는 능력은 안 늘어난다 — 3일차 §3.11-2.")

설정                 학습 토큰      비율  진단
마스킹 O                 71   10.7%  정답만 학습
마스킹 X                661  100.0%  거의 전부가 학습 구간입니다 — 프롬프트 마스킹이 적용되지 않은 것 같습니다.

마스킹을 안 하면 손실의 대부분이 '지문을 그대로 다시 쓰는 법'에서 나온다.
손실 곡선은 더 예쁘게 내려가지만 원하는 능력은 안 늘어난다 — 3일차 §3.11-2.


---
## 5. 품질 점검 5항목 (§3.9-5)

각 항목이 무엇을 막는지 함께 읽으십시오.

In [12]:
import numpy as np
from collections import Counter

def norm_key(rec):
    """근중복 판정 키 — 공백·구두점을 지운 지문+질문."""
    s = (rec["context"] + rec["question"]).lower()
    s = re.sub(r"[^\w가-힣]", "", s)
    return hashlib.md5(s.encode()).hexdigest()

def audit(train, evalset, max_len):
    report = {}

    # ① 중복·근중복
    keys = [norm_key(r) for r in train]
    dup = len(keys) - len(set(keys))
    report["1_duplicates"] = {"n": dup, "pct": round(100 * dup / max(len(train), 1), 2),
                              "위험": "중복이 많으면 그 패턴에 과적합된다"}

    # ② train/eval 누수  ← 가장 치명적
    ek = set(norm_key(r) for r in evalset)
    leak = [i for i, k in enumerate(keys) if k in ek]
    report["2_leakage"] = {"n": len(leak), "pct": round(100 * len(leak) / max(len(train), 1), 2),
                           "위험": "★ 평가 점수가 부풀려진다. 0이어야 한다"}

    # ③ 길이 이상치
    lens = []
    for r in train:
        m = chat.build_messages(r["question"], system=C.SYSTEM_PROMPT, context=r["context"])
        e = chat.build_example(tok, m, make_target(r), max_len=10**6)
        lens.append(e["n_total"])
    lens = np.array(lens)
    over = int((lens > max_len).sum())
    report["3_length"] = {"over_max_len": over, "pct": round(100 * over / max(len(train), 1), 2),
                          "p95": float(np.percentile(lens, 95)), "max": int(lens.max()),
                          "위험": "잘린 샘플은 EOS를 못 배운다 → 응답이 끝나지 않는다"}

    # ④ 응답 형식 불일치
    bad = [r for r in train if not r["answer"].strip() or len(r["answer"]) > 200]
    report["4_format"] = {"n": len(bad), "위험": "빈 정답·과도하게 긴 정답은 학습을 흐린다"}

    # ⑤ 평가셋 오염 (eval 자체의 중복)
    ekeys = [norm_key(r) for r in evalset]
    edup = len(ekeys) - len(set(ekeys))
    report["5_eval_contamination"] = {"eval_duplicates": edup,
                                      "위험": "같은 문항이 여러 번 채점되면 지표가 왜곡된다"}
    return report, lens, set(leak)

report, lens, leak_idx = audit(train_rec, eval_rec, MAX_LEN)
for k, v in report.items():
    print(f"\n{k}")
    for kk, vv in v.items():
        print(f"    {kk:<16}{vv}")


1_duplicates
    n               10
    pct             0.25
    위험              중복이 많으면 그 패턴에 과적합된다

2_leakage
    n               0
    pct             0.0
    위험              ★ 평가 점수가 부풀려진다. 0이어야 한다

3_length
    over_max_len    309
    pct             7.72
    p95             832.0
    max             2309
    위험              잘린 샘플은 EOS를 못 배운다 → 응답이 끝나지 않는다

4_format
    n               0
    위험              빈 정답·과도하게 긴 정답은 학습을 흐린다

5_eval_contamination
    eval_duplicates 1
    위험              같은 문항이 여러 번 채점되면 지표가 왜곡된다


In [13]:
# 걸러내기 — 점검만 하고 넘어가면 점검이 아니다
def filter_records(recs, max_len, exclude_keys=frozenset()):
    seen, kept, dropped = set(), [], Counter()
    for i, r in enumerate(recs):
        k = norm_key(r)
        if k in seen:
            dropped["중복"] += 1; continue
        if k in exclude_keys:
            dropped["eval누수"] += 1; continue
        if not r["answer"].strip() or len(r["answer"]) > 200:
            dropped["형식이상"] += 1; continue
        m = chat.build_messages(r["question"], system=C.SYSTEM_PROMPT, context=r["context"])
        e = chat.build_example(tok, m, make_target(r), max_len=10**6)
        if e["n_total"] > max_len:
            dropped["길이초과"] += 1; continue
        seen.add(k); kept.append(r)
    return kept, dropped

eval_keys = set(norm_key(r) for r in eval_rec)
train_clean, drop_tr = filter_records(train_rec, MAX_LEN, exclude_keys=eval_keys)
eval_clean, drop_ev = filter_records(eval_rec, MAX_LEN)

train_final = train_clean[:C.N_TRAIN]
eval_final = eval_clean[:C.N_EVAL]

print("train 제외 내역:", dict(drop_tr))
print("eval  제외 내역:", dict(drop_ev))
print(f"\n최종: train {len(train_final)}건 / eval {len(eval_final)}건")
assert len(train_final) >= 200, "학습 데이터가 너무 적습니다. N_TRAIN 또는 max_seq_len을 조정하십시오."
assert not (set(norm_key(r) for r in train_final) & set(norm_key(r) for r in eval_final)), \
    "누수가 남아 있습니다"
print("누수 재검증: 통과 (교집합 0)")

train 제외 내역: {'길이초과': 309, '중복': 9}
eval  제외 내역: {'길이초과': 7, '중복': 1}

최종: train 2000건 / eval 200건
누수 재검증: 통과 (교집합 0)


---
## 6. 자기 도메인 데이터 30건 — **직접 작성** (권고 25분)

### 작성 요령
- 자기 업무에서 실제로 자주 받는 질문을 쓴다
- 지문은 사내 문서·공지·매뉴얼의 한 단락(공개 가능한 것만)
- 정답은 **지문 안에 있는 표현**으로 쓴다
- 30건이 어려우면 15건이라도 쓴다. **건너뛰지 마십시오.**

> 사내 비공개 정보는 넣지 마십시오. 이 파일은 저장소에 커밋됩니다.

In [14]:
# 여기에 자기 도메인 데이터를 채우십시오. 예시 2건이 들어 있습니다.
MY_DOMAIN = [
    {
        "context": "연차 휴가는 입사 1년 미만 직원의 경우 매월 개근 시 1일이 발생하며, "
                   "1년 이상 근속한 직원에게는 연 15일이 부여된다. 미사용 연차는 다음 해로 "
                   "이월되지 않으며 수당으로 정산한다.",
        "question": "입사 8개월 된 직원이 받을 수 있는 연차는 최대 몇 일입니까?",
        "answer": "8일",
    },
    {
        "context": "장비 반출 신청은 사용 예정일 3영업일 전까지 그룹웨어에서 제출해야 하며, "
                   "팀장 승인 후 총무팀의 최종 확인을 받아야 효력이 발생한다.",
        "question": "장비 반출 신청은 언제까지 제출해야 합니까?",
        "answer": "사용 예정일 3영업일 전까지",
    },
    # ... 여기에 계속 추가하십시오 (목표 30건)
]

print(f"작성된 건수: {len(MY_DOMAIN)} / 목표 {C.N_DOMAIN}")
if len(MY_DOMAIN) < 15:
    print("[주의] 15건 미만입니다. 3일차 정성 평가의 표본이 부족해집니다.")

작성된 건수: 2 / 목표 30
[주의] 15건 미만입니다. 3일차 정성 평가의 표본이 부족해집니다.


In [15]:
# 자기 데이터도 같은 절차로 점검한다 — 예외를 두지 않는다
domain_rec = []
for i, d in enumerate(MY_DOMAIN):
    ctx, q, ans = clean_text(d["context"]), clean_text(d["question"]), clean_text(d["answer"])
    if ans not in ctx:
        print(f"  [경고] {i+1}번: 정답 {ans!r} 이 지문에 없습니다. "
              f"모델이 지문 밖에서 답을 만들도록 가르치게 됩니다.")
    sents = re.split(r"(?<=[.!?])\s+", ctx)
    ev = next((s for s in sents if ans in s), ctx[:200])
    domain_rec.append({"id": f"domain-{i+1:03d}", "context": ctx, "question": q,
                       "answer": ans, "answer_all": ans, "evidence": ev.strip()})

print(f"\n{len(domain_rec)}건 준비 완료")

  [경고] 1번: 정답 '8일' 이 지문에 없습니다. 모델이 지문 밖에서 답을 만들도록 가르치게 됩니다.

2건 준비 완료


---
## 7. 저장 — `420`·`430`이 읽는다

In [16]:
def to_sft_row(r):
    """학습·평가 공통 형식. 토크나이즈는 420에서 한다(토크나이저가 바뀔 수 있으므로)."""
    return {
        "id": r["id"],
        "system": C.SYSTEM_PROMPT,
        "context": r["context"],
        "question": r["question"],
        "target": make_target(r),
        "answer": r["answer"],
        "answer_all": r["answer_all"],
    }

art.save_jsonl(art.data_path("train.jsonl"), [to_sft_row(r) for r in train_final])
art.save_jsonl(art.data_path("eval.jsonl"), [to_sft_row(r) for r in eval_final])
art.save_jsonl(art.data_path("domain30.jsonl"), [to_sft_row(r) for r in domain_rec])

data_report = {
    "model_id": C.MODEL_ID, "dataset": C.DATASET_ID,
    "max_seq_len_used": MAX_LEN,
    "max_seq_len_source": "020/token_cost.json" if tc else "config 기본값",
    "counts": {"train": len(train_final), "eval": len(eval_final), "domain": len(domain_rec)},
    "audit": report,
    "dropped": {"train": dict(drop_tr), "eval": dict(drop_ev)},
    "length": {"p50": float(np.percentile(lens, 50)), "p95": float(np.percentile(lens, 95)),
               "max": int(lens.max())},
    "leakage_verified_zero": True,
    "task": {"system": C.SYSTEM_PROMPT, "required_fields": C.REQUIRED_FIELDS,
             "output_format": "JSON {answer, evidence}"},
}
art.save_json(art.data_path("data_report.json"), data_report)

for name in ["train.jsonl", "eval.jsonl", "domain30.jsonl", "data_report.json"]:
    p = art.data_path(name)
    print(f"  {p}  ({p.stat().st_size:,} bytes)")

  /content/LLM_FineTuning_Lecture/artifacts/data/train.jsonl  (3,659,345 bytes)
  /content/LLM_FineTuning_Lecture/artifacts/data/eval.jsonl  (372,341 bytes)
  /content/LLM_FineTuning_Lecture/artifacts/data/domain30.jsonl  (1,886 bytes)
  /content/LLM_FineTuning_Lecture/artifacts/data/data_report.json  (1,606 bytes)


In [17]:
# 최종 확인 — 실제 학습에 들어갈 첫 건을 다시 렌더링해서 본다
row = art.load_jsonl(art.data_path("train.jsonl"))[0]
m = chat.build_messages(row["question"], system=row["system"], context=row["context"])
e = chat.build_example(tok, m, row["target"], max_len=MAX_LEN)
print(chat.render_mask(tok, e, max_tokens=260))
print("\n점검:", chat.check_example(tok, e))

■ 회색 = 손실 제외(프롬프트, 590토큰)   ■ 파랑 = 학습됨(정답+EOS, 71토큰)

... (이하 401 토큰 생략)<|im_start|>system
당신은 주어진 지문에서만 근거를 찾아 답하는 한국어 질의응답 도우미입니다. answer와 evidence 두 개의 키를 가진 JSON 객체만 출력하십시오. 설명이나 인사말을 덧붙이지 마십시오.<|im_end|>
<|im_start|>user
1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트

점검: {'ok': True, 'problems': [], 'trainable_tokens': 71, 'total_tokens': 661}


---
## 정리

| 점검 항목 | 결과 | 무엇을 막았는가 |
|---|---|---|
| 중복 | 위 리포트 | 특정 패턴 과적합 |
| **train/eval 누수** | **0 (재검증 완료)** | **부풀려진 평가 점수** |
| 길이 초과 | 제외 처리 | EOS 미학습 → 응답이 안 끝남 |
| 형식 이상 | 제외 처리 | 흐린 학습 신호 |
| eval 오염 | 위 리포트 | 왜곡된 지표 |

### 제출
`artifacts/data/` 4개 파일. 특히 `domain30.jsonl`은 **본인이 작성한 것**이어야 한다.

### 다음
`420`에서 이 데이터로 QLoRA 학습을 돌린다. 3일차 2교시에 학습을 걸고
3교시 강의 중에 돌린 뒤 4교시에 결과를 본다.